In [2]:
from datasets import load_dataset

dataset = load_dataset("csv", 
                       data_files="https://raw.githubusercontent.com/sismetanin/rureviews/master/women-clothing-accessories.3-class.balanced.csv",
                       sep="\t")

dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 90000
    })
})

In [3]:
dataset['train'].column_names

['review', 'sentiment']

In [4]:
dataset['train'].features

{'review': Value('large_string'), 'sentiment': Value('large_string')}

In [19]:
dataset['train'][0]

{'review': 'качество плохое пошив ужасный (горловина наперекос) Фото не соответствует Ткань ужасная рисунок блеклый маленький рукав не такой УЖАС!!!!! не стоит за такие деньги г.......',
 'sentiment': 'negative'}

In [20]:
len(dataset['train'])

90000

## EDA

In [5]:
import pandas as pd

df = dataset["train"].to_pandas()

df.head()

,review,sentiment
0,качество плохое пошив ужасный (горловина напер...,negative
1,"Товар отдали другому человеку, я не получила п...",negative
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative
3,"товар не пришел, продавец продлил защиту без м...",negative
4,"Кофточка голая синтетика, носить не возможно.",negative


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90000 entries, 0 to 89999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     90000 non-null  str  
 1   sentiment  90000 non-null  str  
dtypes: str(2)
memory usage: 22.0 MB


In [7]:
df["sentiment"].value_counts()

sentiment
negative    30000
neautral    30000
positive    30000
Name: count, dtype: int64

In [8]:
df["sentiment"].value_counts(normalize=True)

sentiment
negative    0.333333
neautral    0.333333
positive    0.333333
Name: proportion, dtype: float64

In [10]:
df.isna().sum()

review       0
sentiment    0
dtype: int64

In [12]:
(df["review"].str.strip() == "").sum()

np.int64(1)

In [14]:
df["review"].duplicated().sum()

np.int64(2679)

In [15]:
df["review"].nunique()

87321

In [25]:
df.groupby("sentiment").describe()

review                             
           count unique              top freq
sentiment                                    
neautral   30000  29119                .   39
negative   30000  29218  товар не пришел   47
positive   30000  29413            супер   40

In [34]:
for sentiment in ["negative", "neautral", "positive"]:
    print(f"\n===== {sentiment.upper()} =====")
    
    samples = df[df["sentiment"] == sentiment].sample(3, random_state=1)
    
    for text in samples["review"]:
        print(f"-{text}")


===== NEGATIVE =====
-заказывала 28.05, пришёл заказ 9.08 это ужас полный,лето кончилось....а мне платья пришли!!! долго,очень долго. Заказ не отслеживался! но продавец всегда отвечал на мои вопросы . Платья пришли все в торчащих нитках,печально!!! и легенцы хорошие
-товар так и не дождалась с мая месяца, открывала спор-продливали срок, так и не пришло, деньги вернули
-ГЛАВНОЕ - ТОН ЦВЕТА МРАЧНЫЙ, СИНЕ-ЗЕЛЕНЫЙ, А  НЕ ЯРКИЙ И НАРЯДНЫЙ БИРЮЗОВЫЙ.
ПОСАДКА МЕШКОМ (в груди впритык, в спинка некрасиво вертикально (не учтено в глубине вытачек). В бедрах широковато. Фактически, по верху изделия ширина по проймамам (XXL) - 52*2, по бедрам - 54*2.
НУ ОЧЕНЬ ЖАЛЬ. ПЕРВЫЙ ЗАКАЗ (((

Из плюсов - быстрая отправка и доставка и (2) - легкий наполнитель, БЫСТРЫЙ ОТКЛИК продавца при заказе.
Упаковка обычная, в полиэтиленовый пакет.

===== NEAUTRAL =====
-Большеваты и тонкие,ткань надо поплотнее.Очень быстро дошли и аккуратно сложены в пакете.А так красивые!
-Кофта не приятна к телу и не тянется, надеяла

### Проверим дубликаты по меткам

In [53]:
duplicates = df[df["review"].duplicated(keep=False)].sort_values("review")

duplicates.head(20)

,review,sentiment
30580,!,neautral
41400,!,neautral
54426,!,neautral
86378,!,positive
8511,!!!,negative
46354,!!!,neautral
44364,(,neautral
51141,(,neautral
59655,(,neautral
62373,+,positive


In [47]:
duplicates.groupby("review")["sentiment"].nunique().value_counts()

sentiment
1    557
2    347
3     41
Name: count, dtype: int64

In [50]:
conflicting_duplicates = (duplicates.groupby("review")["sentiment"].nunique())

conflicting_duplicates = conflicting_duplicates[conflicting_duplicates > 1]

len(conflicting_duplicates)

388

In [52]:
for review in conflicting_duplicates.index[:10]:
    print("TEXT:", review)
    print(duplicates[duplicates["review"] == review][
              ["review", "sentiment"]
          ].to_string(index=False)
    )
    
    print("-" * 80)

TEXT: !
review sentiment
     !  neautral
     !  neautral
     !  neautral
     !  positive
--------------------------------------------------------------------------------
TEXT: !!!
review sentiment
   !!!  negative
   !!!  neautral
--------------------------------------------------------------------------------
TEXT: -
review sentiment
     -  negative
     -  negative
     -  negative
     -  negative
     -  negative
     -  neautral
     -  neautral
     -  neautral
     -  neautral
     -  neautral
     -  neautral
     -  neautral
     -  neautral
--------------------------------------------------------------------------------
TEXT: .
review sentiment
     .  negative
     .  negative
     .  negative
     .  negative
     .  negative
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  neautral
     .  n